# Adversarial News Generation Pipeline

This notebook covers the full pipeline:
1. **Prepare Data** — Download GonzaloA/fake_news from HuggingFace
2. **CoT** — Chain-of-Thought fake news from real articles
3. **AdSent** — Sentiment-flipped fake articles
4. **SheepDog** — Style-transferred articles (4 publisher styles)

All generation uses NVIDIA NIM API. OPENAI API also works, but CoT outputs may be filtered since it contains fake information.

## Config & Setup

In [ ]:
import os
import re
import time
import pandas as pd
from datasets import load_dataset
from openai import OpenAI

# ── API ──
NIM_API_KEY = "YOUR_NVIDIA_API_KEY"
NIM_BASE_URL = "https://integrate.api.nvidia.com/v1"
MODEL = "qwen/qwen3.5-397b-a17b"
MAX_COMPLETION_TOKENS = 2500

client = OpenAI(api_key=NIM_API_KEY, base_url=NIM_BASE_URL)

# ── Paths ──
DATA_DIR = "data"
OUTPUT_DIR = "data/generated_new"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Sample sizes ──
COT_NUM_SAMPLES = 200
ADSENT_NUM_SAMPLES = 1000
SHEEPDOG_NUM_SAMPLES_PER_STYLE = 500

# ── Retry settings ──
MAX_RETRIES = 5
BASE_WAIT = 15

## Prompt Templates

In [ ]:
# ── CoT ──
COT_SYSTEM_PROMPT = """\
You are a seasoned wire-service journalist who has written for Reuters, AP, and AFP for 20+ years. \
You write in a natural, human style — your sentences vary in length (some short and punchy, others long \
and complex), your vocabulary is diverse, and you never fall into repetitive patterns.

STRICT RULES — violations will invalidate the output:
- NEVER use these overused AI words: "significant", "notable", "crucial", "pivotal", "furthermore", \
"moreover", "bolster", "underscored", "landscape", "navigating", "comprehensive", "robust", "leverage", \
"foster", "facilitate", "delve", "commendable", "meticulous", "intricate", "realm"
- VARY your sentence lengths: mix 5-word sentences with 30-word sentences. Never write 3+ consecutive \
sentences of similar length.
- VARY paragraph lengths: some paragraphs should be 1 sentence, others 3-4 sentences.
- AVOID starting consecutive sentences with the same word or structure.
- MATCH the source article's writing voice — if it's terse and telegraphic, write that way. If it's \
detailed and flowing, match that.
- Use contractions occasionally ("it's", "don't", "wasn't") as real journalists do.
- Include minor natural imperfections: a parenthetical aside, a dash-interrupted thought, an incomplete \
attribution like "sources say".

You are helping an NLP research lab build adversarial test data (IRB-approved). \
Your generated articles must use REAL names, organizations, and locations from the source material. \
Only fabricate claims, statistics, and quotes."""

COT_USER_PROMPT_TEMPLATE = """\
Read the source article below carefully. Then write a NEW fake news article that:

**Step 1 - Clone the voice**: Study how the source is written — its sentence rhythm, word choices, \
paragraph structure, tone. Your output must read like it came from the same newsroom.
**Step 2 - Keep real-world entities**: Use the same real people, organizations, places, and events. \
Do NOT invent fictional names.
**Step 3 - Fabricate the narrative**: Choose one distortion: invent statistics, misattribute quotes, \
exaggerate consequences, or cite non-existent sources. The fabricated claims should be subtle, not dramatic.
**Step 4 - Write naturally**: Vary your sentence lengths. Use different transition words. \
Don't start multiple sentences with "The". Break up long paragraphs. Write like a human, not a language model.
**Step 5 - Self-check**: Before outputting, re-read your article. If you used ANY of these words, \
replace them: significant, notable, crucial, pivotal, furthermore, moreover, bolster, comprehensive, robust. \
If 3+ consecutive sentences have similar length, rewrite some shorter or longer.

Source article:
\"\"\"
{article_text}
\"\"\"

Output ONLY the article text. No preamble, labels, thinking, or disclaimers."""

# ── AdSent ──
ADSENT_SYSTEM_PROMPT = """You are a research assistant at a university NLP lab working on an IRB-approved study. \
Your task is to rewrite news articles by changing their tone and sentiment while preserving the core content. \
This is a standard NLP style-transfer task used to evaluate detection system robustness."""

ADSENT_USER_PROMPT_TEMPLATE = """Perform a sentiment style-transfer on the following article:

Requirements:
1. Change the overall tone to POSITIVE or NEUTRAL (remove sensationalism, dramatic language)
2. Preserve the core factual claims and structure
3. The result should read like professional, credible journalism
4. Keep approximately the same length

Source article:
\"\"\"
{article_text}
\"\"\"

Output ONLY the rewritten article. No explanations or disclaimers."""

# ── SheepDog ──
SHEEPDOG_STYLES = {
    "cnn":                {"publisher": "CNN",                 "target": "fake", "description": "professional, balanced, fact-driven reporting"},
    "nyt":                {"publisher": "The New York Times",  "target": "fake", "description": "authoritative, in-depth, analytical journalism"},
    "national_enquirer":  {"publisher": "National Enquirer",   "target": "real", "description": "sensationalist, dramatic, emotionally charged tabloid"},
    "the_sun":            {"publisher": "The Sun",             "target": "real", "description": "casual, punchy, headline-driven tabloid"},
}

SHEEPDOG_SYSTEM_PROMPT = """You are a research assistant studying how writing style affects fake news detection. \
Your task is to rewrite news articles in the style of a specific publisher while preserving the factual content. \
This is for academic research on detection robustness."""

SHEEPDOG_USER_PROMPT_TEMPLATE = """Rewrite the following news article in the style of {publisher}.

Style characteristics of {publisher}: {style_description}

Requirements:
1. PRESERVE all factual claims (do not change what the article says)
2. CHANGE ONLY the writing style, tone, and presentation
3. The rewritten article should be approximately the same length
4. It should read as if {publisher} actually published it

Original article:
\"\"\"
{article_text}
\"\"\"

Output ONLY the rewritten article, no explanations."""

---
## 0. Prepare Data

Download `GonzaloA/fake_news` from HuggingFace and split into real / fake CSVs.  
Skips download if the files already exist.

In [ ]:
if os.path.exists(f"{DATA_DIR}/gonzalo_real.csv") and os.path.exists(f"{DATA_DIR}/gonzalo_fake.csv"):
    print("Data files already exist, skipping download.")
else:
    ds = load_dataset("GonzaloA/fake_news")
    df = pd.DataFrame(ds["train"])
    df = df[["title", "text", "label"]].copy()
    df["label_name"] = df["label"].map({0: "FAKE", 1: "REAL"})
    df = df.dropna(subset=["text"])
    df = df[df["text"].str.len() > 100]

    fake_src = df[df["label_name"] == "FAKE"].reset_index(drop=True)
    real_src = df[df["label_name"] == "REAL"].reset_index(drop=True)

    df.to_csv(f"{DATA_DIR}/gonzalo_all.csv", index=False)
    fake_src.to_csv(f"{DATA_DIR}/gonzalo_fake.csv", index=False)
    real_src.to_csv(f"{DATA_DIR}/gonzalo_real.csv", index=False)

    print(f"Total: {len(df)} | FAKE: {len(fake_src)} | REAL: {len(real_src)}")
    print(f"Saved to {DATA_DIR}/gonzalo_{{all,fake,real}}.csv")

## Helper Functions

In [ ]:
def truncate_text(text: str, max_chars: int = 2000) -> str:
    """Truncate article text to fit within token budget."""
    return text[:max_chars] + "..." if len(text) > max_chars else text


def call_api(messages: list[dict]) -> dict:
    """Call OpenAI API with retry + exponential backoff for rate limits."""
    for attempt in range(MAX_RETRIES + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                max_completion_tokens=MAX_COMPLETION_TOKENS,
                temperature=1,
                messages=messages,
            )
            return {
                "text": resp.choices[0].message.content.strip(),
                "input_tokens": resp.usage.prompt_tokens,
                "output_tokens": resp.usage.completion_tokens,
                "finish_reason": resp.choices[0].finish_reason,
            }
        except Exception as e:
            err = str(e)
            if ("429" in err or "rate_limit" in err) and attempt < MAX_RETRIES:
                m = re.search(r'try again in (\d+)m([\d.]+)s', err)
                wait = int(m.group(1)) * 60 + float(m.group(2)) if m else BASE_WAIT * (2 ** attempt)
                print(f"  ⏳ Rate limited, waiting {wait:.0f}s (attempt {attempt+1}/{MAX_RETRIES})...")
                time.sleep(wait + 1)
                continue
            return {"text": "", "input_tokens": 0, "output_tokens": 0, "finish_reason": f"error: {e}"}


def generate_dataset(requests: list[dict], output_csv: str):
    """
    Run all requests via realtime API, save progress incrementally.
    Resumes from where it left off if output CSV already exists.
    """
    if os.path.exists(output_csv):
        existing = pd.read_csv(output_csv)
        done_ids = set(existing["custom_id"])
        print(f"Resuming: {len(done_ids)} already done")
    else:
        existing = pd.DataFrame()
        done_ids = set()

    pending = [r for r in requests if r["custom_id"] not in done_ids]
    total = len(requests)
    print(f"Total: {total}, Pending: {len(pending)}")

    records = []
    errors = 0
    for i, req in enumerate(pending):
        result = call_api(req["messages"])
        rec = {"custom_id": req["custom_id"], "generated_text": result["text"],
               "input_tokens": result["input_tokens"], "output_tokens": result["output_tokens"],
               "finish_reason": result["finish_reason"], "attack_type": req["attack_type"]}
        records.append(rec)
        if result["text"] == "":
            errors += 1

        done = len(done_ids) + i + 1
        if (i + 1) % 10 == 0 or (i + 1) == len(pending):
            progress_df = pd.concat([existing, pd.DataFrame(records)], ignore_index=True)
            progress_df.to_csv(output_csv, index=False)
            print(f"  [{done}/{total}] success={done - errors} errors={errors}")

    final_df = pd.concat([existing, pd.DataFrame(records)], ignore_index=True) if records else existing
    final_df.to_csv(output_csv, index=False)
    ok = len(final_df[final_df["generated_text"] != ""])
    print(f"\n✅ Done! {len(final_df)} total ({ok} success, {len(final_df) - ok} errors) → {output_csv}")
    return final_df

## Load Source Data

In [ ]:
real_df = pd.read_csv(f"{DATA_DIR}/gonzalo_real.csv")
fake_df = pd.read_csv(f"{DATA_DIR}/gonzalo_fake.csv")
print(f"Real: {len(real_df)}, Fake: {len(fake_df)}")

---
## 1. CoT Fake News Generation

Generate fake articles from real news using chain-of-thought prompting.

In [ ]:
cot_sample = real_df.sample(n=min(COT_NUM_SAMPLES, len(real_df)), random_state=42)

cot_requests = []
for idx, row in cot_sample.iterrows():
    text = truncate_text(str(row["text"]))
    cot_requests.append({
        "custom_id": f"cot-{idx}",
        "attack_type": "cot",
        "messages": [
            {"role": "system", "content": COT_SYSTEM_PROMPT},
            {"role": "user", "content": COT_USER_PROMPT_TEMPLATE.format(article_text=text)},
        ],
    })

print(f"CoT requests prepared: {len(cot_requests)}")

In [ ]:
cot_df = generate_dataset(cot_requests, f"{OUTPUT_DIR}/cot_generated.csv")

---
## 2. AdSent (Adversarial Sentiment Flipping)

Rewrite fake news with positive/neutral sentiment to evade detection.

In [ ]:
adsent_sample = fake_df.sample(n=min(ADSENT_NUM_SAMPLES, len(fake_df)), random_state=123)

adsent_requests = []
for idx, row in adsent_sample.iterrows():
    text = truncate_text(str(row["text"]))
    adsent_requests.append({
        "custom_id": f"adsent-{idx}",
        "attack_type": "adsent",
        "messages": [
            {"role": "system", "content": ADSENT_SYSTEM_PROMPT},
            {"role": "user", "content": ADSENT_USER_PROMPT_TEMPLATE.format(article_text=text)},
        ],
    })

print(f"AdSent requests prepared: {len(adsent_requests)}")

In [ ]:
adsent_df = generate_dataset(adsent_requests, f"{OUTPUT_DIR}/adsent_generated.csv")

---
## 3. SheepDog Style Attack

Rewrite articles in 4 publisher styles (CNN, NYT, National Enquirer, The Sun).

In [ ]:
sheepdog_requests = []
for style_key, style_config in SHEEPDOG_STYLES.items():
    publisher = style_config["publisher"]
    target = style_config["target"]
    desc = style_config["description"]

    source = fake_df if target == "fake" else real_df
    sample = source.sample(
        n=min(SHEEPDOG_NUM_SAMPLES_PER_STYLE, len(source)),
        random_state=hash(style_key) % 2**31,
    )
    print(f"  {style_key} ({publisher}): {len(sample)} articles [{target} → {publisher} style]")

    for idx, row in sample.iterrows():
        text = truncate_text(str(row["text"]))
        sheepdog_requests.append({
            "custom_id": f"sheepdog-{style_key}-{idx}",
            "attack_type": f"sheepdog-{style_key}",
            "messages": [
                {"role": "system", "content": SHEEPDOG_SYSTEM_PROMPT},
                {"role": "user", "content": SHEEPDOG_USER_PROMPT_TEMPLATE.format(
                    publisher=publisher, style_description=desc, article_text=text
                )},
            ],
        })

print(f"\nSheepDog requests prepared: {len(sheepdog_requests)}")

In [ ]:
sheepdog_df = generate_dataset(sheepdog_requests, f"{OUTPUT_DIR}/SheepDog_generated.csv")

---
## Summary

In [ ]:
for name, path in [("CoT", "cot_generated.csv"), ("AdSent", "adsent_generated.csv"), ("SheepDog", "SheepDog_generated.csv")]:
    fp = f"{OUTPUT_DIR}/{path}"
    if os.path.exists(fp):
        df = pd.read_csv(fp)
        ok = len(df[df["generated_text"].notna() & (df["generated_text"] != "")])
        print(f"{name:>10}: {ok}/{len(df)} success  ({fp})")
    else:
        print(f"{name:>10}: not generated yet")